# 01 - EDA: Dự đoán sinh viên có nguy cơ bỏ học
FIT4013 - Đề tài 8. Người làm: TV1.

Chạy notebook từ thư mục `notebooks/`. Hình được lưu vào `reports/figures/`.

In [ ]:
import sys
sys.path.append("../src")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from data import load_data, TARGET

sns.set_theme(style="whitegrid")
FIG = Path("../reports/figures")
FIG.mkdir(parents=True, exist_ok=True)

def save(name):
    plt.tight_layout()
    plt.savefig(FIG / name, dpi=150, bbox_inches="tight")
    plt.show()

df = load_data()
df.head()

## 1. Cấu trúc dữ liệu

In [ ]:
print("Kích thước:", df.shape)
print("Số giá trị thiếu:", df.isna().sum().sum())
print("Số dòng trùng:", df.duplicated().sum())
print(df.dtypes.value_counts())
df.info()

## 2. Phân loại cột
Dùng số giá trị khác nhau (`nunique`) để chia cột thành nhị phân, mã định danh và số liên tục.

In [ ]:
nu = df.drop(columns=TARGET).nunique().sort_values()
print(nu.to_string())

In [ ]:
# Chốt danh sách cột (kiểm tra lại với bảng nunique ở trên)
BINARY_COLS = [
    "Daytime/evening attendance", "Displaced", "Educational special needs",
    "Debtor", "Tuition fees up to date", "Gender", "Scholarship holder",
    "International",
]
ID_COLS = [
    "Marital status", "Application mode", "Course", "Previous qualification",
    "Nacionality", "Mother's qualification", "Father's qualification",
    "Mother's occupation", "Father's occupation",
]
feature_cols = [c for c in df.columns if c != TARGET]
NUM_COLS = [c for c in feature_cols if c not in BINARY_COLS + ID_COLS]

# Kiểm tra các cột có tồn tại không
missing = [c for c in BINARY_COLS + ID_COLS if c not in df.columns]
print("Cột không tìm thấy:", missing)
print(f"Nhị phân: {len(BINARY_COLS)} | Định danh: {len(ID_COLS)} | Số: {len(NUM_COLS)}")
print("\nCột số:", NUM_COLS)

**Ghi chú cho báo cáo:** các cột mã định danh (Course, Application mode, nghề cha mẹ...) là mã số không có thứ tự, nên không được coi là biến liên tục. Ở Luồng B chúng được One-Hot.

## 3. Phân phối nhãn

In [ ]:
counts = df[TARGET].value_counts()
pct = (counts / len(df) * 100).round(1)
print(pd.DataFrame({"số lượng": counts, "tỷ lệ %": pct}))

fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df, x=TARGET, order=counts.index, ax=ax)
for p, v in zip(ax.patches, pct):
    ax.annotate(f"{v}%", (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom")
ax.set_title("Phân phối nhãn Target")
ax.set_xlabel("Nhãn"); ax.set_ylabel("Số sinh viên")
save("eda_01_target.png")

Dữ liệu **mất cân bằng nhẹ** (Graduate chiếm gần một nửa, Enrolled chỉ khoảng 18%), vì vậy dùng **F1-macro** làm thang đo chính và chia dữ liệu có `stratify`.

## 4. Biến số: histogram và boxplot

In [ ]:
key_num = [
    "Age at enrollment",
    "Admission grade",
    "Curricular units 1st sem (approved)",
    "Curricular units 1st sem (grade)",
    "Curricular units 2nd sem (approved)",
    "Curricular units 2nd sem (grade)",
]
key_num = [c for c in key_num if c in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, c in zip(axes.ravel(), key_num):
    sns.histplot(df[c], bins=30, ax=ax)
    ax.set_title(c, fontsize=10)
fig.suptitle("Histogram các biến số chính")
save("eda_02_hist.png")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, c in zip(axes.ravel(), key_num):
    sns.boxplot(data=df, x=TARGET, y=c, ax=ax,
                order=["Dropout", "Enrolled", "Graduate"])
    ax.set_title(c, fontsize=10); ax.set_xlabel("")
fig.suptitle("Boxplot biến số theo nhãn (nhận diện ngoại lệ)")
save("eda_03_boxplot.png")

In [ ]:
# Thống kê ngoại lệ theo quy tắc IQR
rows = []
for c in NUM_COLS:
    q1, q3 = df[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    n_out = ((df[c] < q1 - 1.5 * iqr) | (df[c] > q3 + 1.5 * iqr)).sum()
    rows.append((c, n_out, round(n_out / len(df) * 100, 1)))
pd.DataFrame(rows, columns=["cột", "số ngoại lệ", "%"]).sort_values("số ngoại lệ", ascending=False)

## 5. Biến tài chính và nhân khẩu theo nhãn

In [ ]:
cols = ["Debtor", "Tuition fees up to date", "Scholarship holder", "Gender"]
cols = [c for c in cols if c in df.columns]
fig, axes = plt.subplots(1, len(cols), figsize=(4.5 * len(cols), 4))
for ax, c in zip(axes, cols):
    sns.countplot(data=df, x=c, hue=TARGET, hue_order=["Dropout", "Enrolled", "Graduate"], ax=ax)
    ax.set_title(c)
save("eda_04_financial.png")

In [ ]:
# Tỷ lệ Dropout theo từng nhóm (dễ đọc hơn đếm thô)
for c in cols:
    print(f"\n{c}")
    print((pd.crosstab(df[c], df[TARGET], normalize="index") * 100).round(1))

## 6. Heatmap tương quan

In [ ]:
tmp = df.copy()
tmp["Target_code"] = tmp[TARGET].map({"Dropout": 0, "Enrolled": 1, "Graduate": 2})
corr_cols = NUM_COLS + ["Target_code"]
corr = tmp[corr_cols].corr()

plt.figure(figsize=(13, 10))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False, square=True)
plt.title("Ma trận tương quan các biến số")
save("eda_05_corr.png")

print(corr["Target_code"].drop("Target_code").sort_values(key=abs, ascending=False).head(10).round(3))

Các cặp biến học kỳ 1 và học kỳ 2 (đăng ký, đạt, điểm) tương quan mạnh với nhau. Điều này gợi ý tạo đặc trưng tỷ lệ ở Luồng B.

## 7. Dữ liệu bất thường

In [ ]:
g1 = "Curricular units 1st sem (grade)"
e1 = "Curricular units 1st sem (enrolled)"
e2 = "Curricular units 2nd sem (enrolled)"

print("Điểm HK1 = 0:", (df[g1] == 0).sum())
print("Số môn đăng ký HK1 = 0:", (df[e1] == 0).sum())
print("Số môn đăng ký HK2 = 0:", (df[e2] == 0).sum())

zero_grade = df[df[g1] == 0]
print("\nPhân bố nhãn của nhóm điểm HK1 = 0:")
print((zero_grade[TARGET].value_counts(normalize=True) * 100).round(1))

zero_enr = df[df[e1] == 0]
print("\nPhân bố nhãn của nhóm 0 môn đăng ký HK1:")
print((zero_enr[TARGET].value_counts(normalize=True) * 100).round(1))

**Nhận xét cho báo cáo:** điểm bằng 0 thường là sinh viên không đăng ký hoặc không thi, không phải lỗi nhập liệu. Nhóm này nghiêng mạnh về Dropout nên là tín hiệu hữu ích, **giữ nguyên, không xóa hay điền bù**. Khi tính tỷ lệ ở Luồng B, các dòng 0 môn đăng ký cho kết quả 0 (chia cho 0 trả về 0).

## 8. Tóm tắt kết quả EDA (điền số thật vào báo cáo)

In [ ]:
print(f"Kích thước: {df.shape}, thiếu: {df.isna().sum().sum()}, trùng: {df.duplicated().sum()}")
print("Nhãn:", pct.to_dict())
print("Danh sách hình đã lưu:")
for f in sorted(FIG.glob("eda_*.png")):
    print(" -", f.name)